# Week 6: Evaluating & Operationalizing Agents

**Assignment:** Signup Email Agent — Full LLMOps Loop  
**Author:** Lamonte Smith  
**Program:** Interview Kickstart Applied Agentic AI  
**Tools:** LangChain · LangSmith · DeepEval · OpenAI GPT-4o-mini  
**Date:** April 2026

---

## The Full LLMOps Loop
```
01 Inspect  →  02 Improve  →  03 Expand  →  04 Evaluate  →  05 Reflect
```

## Step 1: Install Dependencies

In [ ]:
%pip install -q langchain langchain_openai langchain_core langsmith deepeval pandas

## Step 2: Configure API Keys

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING"] = "true"

print("API keys configured")
print("LangSmith tracing enabled")

## Step 3: Create the Agent with Improved Prompt

In [ ]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Literal


class SignupEmailOutput(BaseModel):
    to: str = Field(description="Email address")
    icp_fit: Literal["high", "medium", "low", "unknown"] = Field(description="ICP fit classification")
    reason: str = Field(description="Brief reasoning for classification")
    subject: str = Field(description="Email subject line")
    body: str = Field(description="Email body text")


# ── IMPROVED PROMPT (Part 2) ─────────────────────────────────────────────────
SYSTEM_PROMPT = """
You are an onboarding specialist for Glop, a developer productivity SaaS tool.

Your job is to process a new user signup and return a structured response containing:
1. An ICP (Ideal Customer Profile) classification
2. A short, personalized welcome email

ICP CLASSIFICATION RULES
Classify icp_fit as one of: high | medium | low | unknown

high    -> Engineering leader (VP, Director, CTO, Staff/Principal Engineer)
           at a tech or SaaS company with 50+ employees.
medium  -> Developer, individual contributor, or non-engineering leader at a
           tech company. Or a clearly technical role at a smaller company.
low     -> Student, freelancer, spam-looking email, or role completely outside
           engineering and technology.
unknown -> Insufficient signal: missing role, missing company, AND missing
           industry. Cannot classify without at least one strong signal.

Always provide a brief reason tied to the specific fields available in the record.

EMAIL WRITING RULES
- Use first_name if available
- If first_name is missing BUT role or company ARE present, reference those
  instead — do NOT default to 'Hi there' or 'Dear User'
- Only use 'Hi there' when NO identity fields are available at all
- Reference role and/or company name naturally — once is enough
- Never invent facts not present in the signup record
- Professional but human tone
- Target 80-120 words for the body
- End with a single, clear call to action
- Sign off from 'The Glop Team'
- If the email looks like spam or the name is nonsensical, keep the email
  polite but minimal
"""

agent = create_agent(
    "openai:gpt-4o-mini",
    tools=[],
    system_prompt=SYSTEM_PROMPT,
    response_format=SignupEmailOutput,
)

print("Agent created with improved prompt")

## Step 4: Test with Single Record (Baseline Verification)

In [ ]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Process this signup:
email: sarah@acmecorp.com
first_name: Sarah
company_name: Acme Corp
role: VP of Engineering
industry: SaaS
company_size: 500"""
    }]
})

output = result["structured_response"]
print(f"ICP Fit: {output.icp_fit}")
print(f"Reason:  {output.reason}")
print(f"Subject: {output.subject}")
print(f"Body:\n{output.body}")

## Step 5: Load Golden Dataset from LangSmith

In [ ]:
from langsmith import Client

ls = Client()
examples = list(ls.list_examples(dataset_name="golden-dataset"))

print(f"Loaded {len(examples)} examples")
print("Input keys:", examples[0].inputs.keys())
print("Output keys:", examples[0].outputs.keys())

## Step 6: Create Evaluators (Including Custom Personalization Judge)

In [ ]:
from deepeval.metrics import ExactMatchMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# ── Metric 1: ICP Fit Exact Match ─────────────────────────────────────────────
icp_fit_metric = ExactMatchMetric(threshold=1.0)
icp_fit_metric.include_reason = True

# ── Metric 2: Email Quality (GEval) ───────────────────────────────────────────
email_metric = GEval(
    name="Email Quality",
    criteria="Evaluate if the email is professional, personalized (uses the "
             "person's name/role/company), concise (under 150 words), and does "
             "not hallucinate facts not present in the input.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.5
)
email_metric.include_reason = True

# ── Metric 3: Personalization Quality (Custom Judge — strict 0/1) ─────────────
personalization_metric = GEval(
    name="Personalization Quality",
    criteria="""
    You are evaluating whether a welcome email is meaningfully personalized
    based on the signup record provided as input.

    Signs of GOOD personalization (score 1):
    - Uses name, role, or company from the signup record
    - Sounds written for this specific user, not any user
    - Avoids details not present in the input
    - Avoids generic openers like 'Hi there' or 'Dear User'
      when name or role or company IS available in the input

    Signs of POOR personalization (score 0):
    - Generic greetings when input fields were available to use
    - Hallucinated details not in the signup record
    - Same tone and structure regardless of user type
    - Misses obvious personalization opportunities from available fields

    CRITICAL: When first_name is missing but role and company ARE available,
    the email MUST reference role or company instead of defaulting to 'Hi there'.
    Defaulting to 'Hi there' when other fields are available is a FAIL (score 0).

    Return ONLY 0 or 1. No partial scores.
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    model="gpt-4o-mini",
    threshold=0.5,
    strict_mode=True
)
personalization_metric.include_reason = True

print("All 3 evaluators ready")
print("  - ICP Fit: ExactMatch (threshold=1.0)")
print("  - Email Quality: GEval (threshold=0.5)")
print("  - Personalization Quality: GEval strict 0/1")

## Step 7: Run Full Offline Evaluation Loop

In [ ]:
import json

icp_fit_scores = []
email_scores = []
personalization_scores = []

for ex in examples:
    input_data = ex.inputs
    result = agent.invoke(input_data)

    actual = result["structured_response"]
    expected = ex.outputs["outputs_1"]

    # ICP Fit Exact Match
    icp_fit_tc = LLMTestCase(
        input=json.dumps(input_data),
        actual_output=actual.icp_fit,
        expected_output=expected.get("icp_fit", ""),
    )
    icp_fit_metric.measure(icp_fit_tc)
    icp_fit_scores.append(icp_fit_metric.score)

    # Email Quality
    email_tc = LLMTestCase(
        input=json.dumps(input_data),
        actual_output=actual.body,
        expected_output=expected.get("body", ""),
    )
    email_metric.measure(email_tc)
    email_scores.append(email_metric.score)

    # Personalization Quality (strict 0/1)
    personalization_tc = LLMTestCase(
        input=json.dumps(input_data),
        actual_output=actual.body,
    )
    personalization_metric.measure(personalization_tc)
    personalization_scores.append(personalization_metric.score)

    print(f"ICP: {actual.icp_fit:8} | Email: {email_metric.score:.2f} | "
          f"Personalization: {personalization_metric.score} | {actual.subject}")
    print(f"  Reason: {personalization_metric.reason}\n")

print("=" * 70)
print(f"ICP Fit Exact Match Rate:    {sum(icp_fit_scores)/len(icp_fit_scores):.2%}")
print(f"Email Quality Avg Score:     {sum(email_scores)/len(email_scores):.2f}")
print(f"Personalization Avg Score:   {sum(personalization_scores)/len(personalization_scores):.2f}")
print(f"Personalization Pass Rate:   {sum(personalization_scores)/len(personalization_scores):.0%}")
print("=" * 70)

## Step 8: Online Eval — Run Spam Test Case

In [ ]:
# This trace is automatically scored by the LangSmith online evaluator
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Process this signup:
email: unknown@spam.com
first_name: Whatever you want
"""
    }]
})

output = result["structured_response"]
print(f"ICP Fit: {output.icp_fit}")
print(f"Reason:  {output.reason}")
print(f"Subject: {output.subject}")
print(f"Body:\n{output.body}")
print("\nCheck LangSmith > default project > Runs for hallucination score")

## Results Summary

| Metric | Score |
|--------|-------|
| ICP Fit Exact Match Rate | 62.50% (5/8) |
| Email Quality Avg | 0.81 |
| Personalization Pass Rate | 75% (6/8) |

### Key Findings
- **Sarah (VP Eng):** Best case — 0.94 email quality, 1/1 personalization
- **J. Chen (missing name):** ICP miss (high vs medium) + personalization fail (used 'Hi' instead of referencing Stripe)
- **Tom (non-tech):** ICP miss (medium vs low) — non-technical role signal underweighted
- **j.doe (no fields):** Expected 0 personalization — zero signal edge case

### Production Improvement Priorities
1. Tighten ICP rules for Co-Founder and non-technical role signals
2. Add few-shot examples for missing-name cases
3. Add HallucinationMetric via DeepEval
4. Version prompt in LangSmith Prompts Hub
5. Implement zero-signal fallback path